In [1]:
!pip install transformers
!pip install peft
!pip install torch torchaudio


In [2]:
!cp -r /kaggle/input/datasets/supercoolwizard/test1-2min/data /kaggle/working/

In [3]:
import os
os.mkdir("models")

In [4]:
from pydantic import DirectoryPath
from pydantic_settings import BaseSettings
from pathlib import Path
import torch

class Settings(BaseSettings):
    DATA_DIR: DirectoryPath = Path("data")
    # INPUT_DIR: DirectoryPath = Path("input")
    MODELS_DIR: DirectoryPath = Path("models")

    student_model: str = "UsefulSensors/moonshine-tiny"
    teacher_model: str ="openai/whisper-tiny"

    device: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

settings = Settings()

In [5]:
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = []
        label_features = []
        for feature in features:
            input_features.append({"input_values": feature["input_values"]})
            label_features.append({"input_ids": feature["labels"]})

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels

        return batch

In [ ]:
from transformers import MoonshineForConditionalGeneration, TrainingArguments, Trainer
from transformers import AutoProcessor
from peft import LoraConfig, get_peft_model
from datasets import load_dataset, Audio
import datasets

model = MoonshineForConditionalGeneration.from_pretrained(settings.student_model)
processor = AutoProcessor.from_pretrained(settings.student_model)

processor.tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model.resize_token_embeddings(len(processor.tokenizer))
model.config.pad_token_id = processor.tokenizer.pad_token_id

# model
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj"],
    task_type=None
)

model = get_peft_model(model, config)

# data
dataset = load_dataset("csv", data_files={"train": str(settings.DATA_DIR / "train" / "metadata.csv")})
dataset = dataset.cast_column("audio_path", datasets.Value("string"))
dataset = dataset.cast_column("audio_path", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    audio = batch["audio_path"]
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    )
    # print(inputs.input_values[0])
    batch["input_values"] = inputs.input_values[0]

    with open(batch["teacher_path"], "r", encoding="utf-8") as f:
        transcript = f.read().strip()

    batch["labels"] = processor.tokenizer(transcript).input_ids
    return batch

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names
)
# print(dataset["train"])

# fine_tuning
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

training_args = TrainingArguments(
    output_dir=str(settings.MODELS_DIR),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=50,
    max_steps=100,
    fp16=False,
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=data_collator,
)

trainer.train()
model.save_pretrained(str(settings.MODELS_DIR))